# Phase 3d — Approach 4: EfficientNet-B0 Fine-Tuning

**CSC3109 Machine Learning | Group 30**

## What Is EfficientNet?

EfficientNet (2019) is a family of models that achieves better accuracy with fewer parameters than ResNet by scaling three dimensions together:

| Dimension | What it means |
|-----------|---------------|
| **Depth** | More layers → learns more complex features |
| **Width** | More channels per layer → captures more patterns |
| **Resolution** | Higher input image size → finer detail |

Instead of scaling these independently (which wastes compute), EfficientNet uses a **compound coefficient** to scale all three together optimally.

**EfficientNet-B0** is the smallest variant — ~5.3M parameters vs ResNet-50's 25M — yet matches or exceeds ResNet-50 accuracy on many benchmarks.

**Strategy:** Full fine-tuning — we unfreeze all layers and train end-to-end with a low learning rate.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

from dataset import get_data_loaders
from train_utils import train_model, plot_training_curves, plot_confusion_matrix, \
                        get_all_predictions, print_classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORT_DIR = REPO_ROOT / 'report'
print(f'Device: {DEVICE}')

In [ ]:
train_loader, val_loader, cls2idx, idx2cls = get_data_loaders(
    data_dir=REPO_ROOT / 'data', batch_size=32, num_workers=0
)
NUM_CLASSES = len(cls2idx)

---
## Step 2 — Load EfficientNet-B0 and Replace Classifier

In [ ]:
# Load EfficientNet-B0 with ImageNet pre-trained weights
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# EfficientNet's classifier is model.classifier[1] (a Linear layer)
# in_features is 1280 for B0
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

# All layers are trainable (full fine-tuning)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'EfficientNet-B0 loaded.')
print(f'Classifier input features: {in_features}')
print(f'Trainable parameters: {trainable:,}')
print('Strategy: full fine-tuning (all layers unlocked)')

---
## Step 3 — Configure Training

For full fine-tuning, we use a **low learning rate** (5e-5) across all layers to preserve the pre-trained features while still allowing adaptation.

In [ ]:
NUM_EPOCHS = 30

optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)

print('Optimiser: AdamW (lr=5e-5, weight_decay=1e-4)')
print('Scheduler: Cosine Annealing')
print('AdamW vs Adam: AdamW applies weight decay more correctly — better regularisation')

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_name='efficientnet_b0',
    device=DEVICE
)

In [ ]:
plot_training_curves(
    history,
    title='Approach 4 — EfficientNet-B0 Fine-Tuning',
    save_path=REPORT_DIR / 'approach4_efficientnet_curves.png'
)

In [ ]:
all_labels, all_preds = get_all_predictions(model, val_loader, DEVICE)
print('=== Approach 4: EfficientNet-B0 — Validation Results ===')
print_classification_report(all_labels, all_preds)
print(f'Overall Accuracy: {(all_labels == all_preds).mean():.4f}')

In [ ]:
plot_confusion_matrix(
    all_labels, all_preds,
    title='Approach 4 — EfficientNet-B0',
    save_path=REPORT_DIR / 'approach4_efficientnet_confusion.png'
)

---
## Observations for Report

**Expected outcome:** EfficientNet-B0 is a strong candidate for best performance. It achieves ResNet-50 accuracy with 5× fewer parameters, meaning less risk of overfitting on our small dataset.

**Strengths:** Parameter efficient, strong accuracy, good generalisation, MobileNet-style depthwise separable convolutions reduce overfitting risk  
**Weaknesses:** Slightly more complex architecture to explain; full fine-tuning can overfit if dataset is too small

Best model saved to `models/efficientnet_b0_best.pth`

**Next:** `03e_vit.ipynb`